# Fraud Detection MLOps - Interactive Demonstration

This notebook provides a hands-on walkthrough of all features in the fraud detection MLOps system:
- Data loading and exploration
- Feature selection with SHAP
- Model training with Optuna tuning
- Ensemble evaluation
- Explainability analysis
- API deployment testing
- Monitoring and drift detection

**Time estimate**: 15-30 minutes depending on sample size

## Section 1: Environment Setup and Dependencies

In [ ]:
# Import standard libraries
import sys
import os
from pathlib import Path
import json
import warnings

# Add project root to path
project_root = Path.cwd()
if 'fraud-detection' in str(project_root):
    sys.path.insert(0, str(project_root))
else:
    sys.path.insert(0, '/workspaces/fraud-detection')

warnings.filterwarnings('ignore')

# Data processing
import pandas as pd
import numpy as np

# ML libraries
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Project imports
from src.data_preprocessing import load_config, preprocess_pipeline, load_data
from src.validation import DataValidator
from src.feature_selection import shap_feature_selection
from src.stacking_model import StackingFraudDetector
from src.evaluation import compute_metrics, print_results

print("✓ All imports successful")
print(f"✓ Project root: {project_root}")

In [ ]:
# Load configuration
config = load_config('config/params.yaml')

# Display configuration
print("Configuration:")
print(f"  - Data path: {config['data']['train_transaction']}")
print(f"  - Test size: {config['data']['test_size']}")
print(f"  - Sample size: {config['data'].get('sample_size', 'Full dataset')}")
print(f"  - Optuna trials: {config['optuna']['n_trials']}")
print(f"  - Top features: {config['feature_selection']['n_top_features']}")

# Set style for visualizations
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## Section 2: Load and Explore Dataset

In [ ]:
# Load raw data
print("Loading dataset...")
X_train, X_test, y_train, y_test, feature_names = preprocess_pipeline(config)

print(f"\n✓ Data loaded successfully:")
print(f"  - Training set: {X_train.shape[0]:,} samples, {X_train.shape[1]} features")
print(f"  - Test set: {X_test.shape[0]:,} samples")
print(f"  - Features: {len(feature_names)}")

In [ ]:
# Exploratory Data Analysis
print("\n=== Class Distribution ===")
fraud_rate_train = y_train.sum() / len(y_train) * 100
fraud_rate_test = y_test.sum() / len(y_test) * 100

print(f"\nTraining set:")
print(f"  - Legitimate: {(y_train == 0).sum():,} ({100-fraud_rate_train:.2f}%)")
print(f"  - Fraudulent: {(y_train == 1).sum():,} ({fraud_rate_train:.2f}%)")

print(f"\nTest set:")
print(f"  - Legitimate: {(y_test == 0).sum():,} ({100-fraud_rate_test:.2f}%)")
print(f"  - Fraudulent: {(y_test == 1).sum():,} ({fraud_rate_test:.2f}%)")

In [ ]:
# Feature statistics
print("\n=== Feature Statistics ===")
X_train_df = pd.DataFrame(X_train, columns=feature_names) if isinstance(X_train, np.ndarray) else X_train
print(f"\nFeature count: {X_train_df.shape[1]}")
print(f"\nBasic statistics:")
print(X_train_df.describe().round(3))

In [ ]:
# Data quality checks
print("\n=== Data Quality ===")
print(f"\nMissing values:")
if isinstance(X_train, np.ndarray):
    missing = np.isnan(X_train).sum()
    print(f"  - Total missing: {missing}")
else:
    missing_per_col = X_train.isna().sum()
    print(missing_per_col[missing_per_col > 0])
    print(f"  - Columns with missing: {(missing_per_col > 0).sum()}")

print(f"\nData types: {X_train.dtype if isinstance(X_train, np.ndarray) else X_train.dtypes.value_counts().to_dict()}")

In [ ]:
# Visualize class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Training set
class_counts_train = pd.Series(y_train).value_counts()
axes[0].bar(['Legitimate', 'Fraudulent'], class_counts_train.values, color=['green', 'red'])
axes[0].set_title('Training Set - Class Distribution')
axes[0].set_ylabel('Count')
axes[0].set_yscale('log')

# Test set
class_counts_test = pd.Series(y_test).value_counts()
axes[1].bar(['Legitimate', 'Fraudulent'], class_counts_test.values, color=['green', 'red'])
axes[1].set_title('Test Set - Class Distribution')
axes[1].set_ylabel('Count')
axes[1].set_yscale('log')

plt.tight_layout()
plt.show()

print("✓ Class distribution visualization complete")

## Section 3: Data Preprocessing Pipeline

In [ ]:
# Data validation
print("=== Data Validation ===")
validator = DataValidator()

# Prepare data as DataFrame for validation
X_train_df = pd.DataFrame(X_train, columns=feature_names) if isinstance(X_train, np.ndarray) else X_train
X_test_df = pd.DataFrame(X_test, columns=feature_names) if isinstance(X_test, np.ndarray) else X_test

# Add target column
train_df = X_train_df.copy()
train_df['isFraud'] = y_train

test_df = X_test_df.copy()
test_df['isFraud'] = y_test

# Validate schema
is_valid, errors = validator.validate_schema(train_df)
print(f"\nSchema validation: {'✓ PASSED' if is_valid else '✗ FAILED'}")
if errors:
    print(f"  Errors: {len(errors)}")
    for err in errors[:5]:  # Show first 5 errors
        print(f"    - {err}")
else:
    print(f"  All features validated")

In [ ]:
# Data quality check
print("\n=== Data Quality Assessment ===")
quality_report = validator.validate_data_quality(train_df)

print(f"\nQuality Metrics:")
print(f"  - Completeness: {quality_report.get('completeness', 0):.2%}")
print(f"  - Features with missing values: {quality_report.get('missing_count', 0)}")
print(f"  - Duplicate rows: {quality_report.get('duplicates', 0)")
print(f"  - Average missing per row: {quality_report.get('avg_missing_per_row', 0):.2%}")

if 'missing_by_column' in quality_report:
    print(f"\nTop missing columns:")
    missing_cols = quality_report['missing_by_column']
    for col, pct in sorted(missing_cols.items(), key=lambda x: x[1], reverse=True)[:5]:
        print(f"  - {col}: {pct:.2%}")

In [ ]:
# SMOTE for class balancing
from imblearn.over_sampling import SMOTE

print("\n=== Class Balancing with SMOTE ===")
print(f"\nBefore SMOTE:")
print(f"  - Class 0: {(y_train == 0).sum():,}")
print(f"  - Class 1: {(y_train == 1).sum():,}")
print(f"  - Ratio: {(y_train == 1).sum() / (y_train == 0).sum() * 100:.2f}%")

smote = SMOTE(random_state=42, k_neighbors=5)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

print(f"\nAfter SMOTE:")
print(f"  - Class 0: {(y_train_balanced == 0).sum():,}")
print(f"  - Class 1: {(y_train_balanced == 1).sum():,}")
print(f"  - Ratio: {(y_train_balanced == 1).sum() / (y_train_balanced == 0).sum() * 100:.2f}%")
print(f"\n✓ Data preprocessing complete")

## Section 4: Model Training and Evaluation

In [ ]:
# Feature selection with SHAP
print("=== Feature Selection with SHAP ===")
print("\nTraining base XGBoost for SHAP analysis...")

xgb_base = xgb.XGBClassifier(n_estimators=50, random_state=42)
xgb_base.fit(X_train_balanced, y_train_balanced)

print("✓ Base model trained")

# Get top features using SHAP
print("\nCalculating SHAP feature importances...")
selected_features, selected_indices = shap_feature_selection(
    xgb_base, 
    X_train_balanced, 
    feature_names, 
    n_features=30
)

print(f"\n✓ Top 30 features selected:")
for i, feat in enumerate(selected_features, 1):
    print(f"  {i:2d}. {feat}")

In [ ]:
# Select features for model training
X_train_selected = X_train_balanced[:, selected_indices]
X_test_selected = X_test[:, selected_indices]

print(f"\n=== Selected Feature Set ===")
print(f"Original features: {X_train_balanced.shape[1]}")
print(f"Selected features: {X_train_selected.shape[1]}")
print(f"Reduction: {(1 - X_train_selected.shape[1] / X_train_balanced.shape[1]) * 100:.1f}%")

In [ ]:
# Train ensemble models
print("\n=== Training Ensemble Models ===")

print("\nTraining XGBoost...")
xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=7,
    learning_rate=0.1,
    random_state=42,
    n_jobs=-1
)
xgb_model.fit(X_train_selected, y_train_balanced)
y_pred_xgb = xgb_model.predict_proba(X_test_selected)[:, 1]
print(f"  - Train AUC: {roc_auc_score(y_train_balanced, xgb_model.predict_proba(X_train_selected)[:, 1]):.4f}")
print(f"  - Test AUC: {roc_auc_score(y_test, y_pred_xgb):.4f}")

print("\nTraining LightGBM...")
lgb_model = lgb.LGBMClassifier(
    n_estimators=100,
    max_depth=7,
    learning_rate=0.1,
    random_state=42,
    n_jobs=-1
)
lgb_model.fit(X_train_selected, y_train_balanced)
y_pred_lgb = lgb_model.predict_proba(X_test_selected)[:, 1]
print(f"  - Train AUC: {roc_auc_score(y_train_balanced, lgb_model.predict_proba(X_train_selected)[:, 1]):.4f}")
print(f"  - Test AUC: {roc_auc_score(y_test, y_pred_lgb):.4f}")

print("\nTraining CatBoost...")
cb_model = CatBoostClassifier(
    iterations=100,
    depth=7,
    learning_rate=0.1,
    random_state=42,
    verbose=False
)
cb_model.fit(X_train_selected, y_train_balanced)
y_pred_cb = cb_model.predict_proba(X_test_selected)[:, 1]
print(f"  - Train AUC: {roc_auc_score(y_train_balanced, cb_model.predict_proba(X_train_selected)[:, 1]):.4f}")
print(f"  - Test AUC: {roc_auc_score(y_test, y_pred_cb):.4f}")

In [ ]:
# Train meta-learner for stacking
print("\n=== Training Meta-Learner ===")
print("Creating meta-features from base model predictions...")

# Prepare meta-features for training
meta_train = np.column_stack([
    xgb_model.predict_proba(X_train_selected)[:, 1],
    lgb_model.predict_proba(X_train_selected)[:, 1],
    cb_model.predict_proba(X_train_selected)[:, 1]
])

meta_test = np.column_stack([
    y_pred_xgb,
    y_pred_lgb,
    y_pred_cb
])

print(f"Meta-feature shape: {meta_train.shape}")

# Train meta-learner
print("\nTraining XGBoost meta-learner...")
meta_model = xgb.XGBClassifier(
    n_estimators=50,
    max_depth=5,
    learning_rate=0.1,
    random_state=42
)
meta_model.fit(meta_train, y_train_balanced)

# Final predictions
y_pred_ensemble = meta_model.predict_proba(meta_test)[:, 1]
print(f"✓ Ensemble model trained")

In [ ]:
# Comprehensive evaluation
print("\n=== Model Evaluation ===")

# Individual models
models = {
    'XGBoost': y_pred_xgb,
    'LightGBM': y_pred_lgb,
    'CatBoost': y_pred_cb,
    'Ensemble': y_pred_ensemble
}

results = {}
for name, pred_proba in models.items():
    pred = (pred_proba >= 0.5).astype(int)
    
    acc = accuracy_score(y_test, pred)
    prec = precision_score(y_test, pred, zero_division=0)
    rec = recall_score(y_test, pred, zero_division=0)
    f1 = f1_score(y_test, pred, zero_division=0)
    auc = roc_auc_score(y_test, pred_proba)
    
    results[name] = {
        'accuracy': acc,
        'precision': prec,
        'recall': rec,
        'f1': f1,
        'auc': auc
    }

# Display results
results_df = pd.DataFrame(results).T
print("\nPerformance Metrics:")
print(results_df.round(4))

print(f"\n✓ Best model: {results_df['auc'].idxmax()} (AUC: {results_df['auc'].max():.4f})")

In [ ]:
# Confusion matrices
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for idx, (name, pred_proba) in enumerate(models.items()):
    pred = (pred_proba >= 0.5).astype(int)
    cm = confusion_matrix(y_test, pred)
    
    im = axes[idx].imshow(cm, cmap='Blues')
    axes[idx].set_title(f'{name} - Confusion Matrix')
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('Actual')
    
    # Add text annotations
    for i in range(2):
        for j in range(2):
            axes[idx].text(j, i, str(cm[i, j]), ha='center', va='center', color='white')
    
    axes[idx].set_xticks([0, 1])
    axes[idx].set_yticks([0, 1])
    axes[idx].set_xticklabels(['0', '1'])
    axes[idx].set_yticklabels(['0', '1'])
    
    plt.colorbar(im, ax=axes[idx])

plt.tight_layout()
plt.show()

print("✓ Confusion matrices visualization complete")

## Section 5: Model Deployment Configuration

In [ ]:
print("=== Model Deployment Configuration ===")

# Create model package
model_metadata = {
    'model_type': 'stacking_ensemble',
    'models': ['xgboost', 'lightgbm', 'catboost'],
    'meta_learner': 'xgboost',
    'selected_features': selected_features.tolist() if isinstance(selected_features, np.ndarray) else selected_features,
    'feature_count': len(selected_features),
    'training_samples': X_train_balanced.shape[0],
    'test_samples': X_test.shape[0],
    'metrics': results['Ensemble'],
    'version': '1.0.0',
    'created_at': pd.Timestamp.now().isoformat()
}

print("\nModel Package Configuration:")
for key, value in model_metadata.items():
    if key != 'selected_features':
        print(f"  {key}: {value}")
    else:
        print(f"  {key}: {len(value)} features")

# Save metadata
metadata_path = 'models/metadata.json'
os.makedirs('models', exist_ok=True)
with open(metadata_path, 'w') as f:
    # Convert numpy types to Python types for JSON serialization
    metadata_safe = model_metadata.copy()
    metadata_safe['metrics'] = {k: float(v) for k, v in metadata_safe['metrics'].items()}
    json.dump(metadata_safe, f, indent=2)

print(f"\n✓ Model metadata saved to {metadata_path}")

In [ ]:
# Create inference wrapper
print("\n=== Creating Inference Wrapper ===")

class SimpleEnsamblePredictor:
    """Simple wrapper for the trained ensemble model."""
    
    def __init__(self, base_models, meta_model, feature_indices):
        self.xgb = base_models['xgb']
        self.lgb = base_models['lgb']
        self.cb = base_models['cb']
        self.meta = meta_model
        self.feature_indices = feature_indices
    
    def predict(self, X):
        """Make predictions."""
        X_selected = X[:, self.feature_indices] if isinstance(X, np.ndarray) else X.iloc[:, self.feature_indices]
        
        # Get base model predictions
        xgb_pred = self.xgb.predict_proba(X_selected)[:, 1]
        lgb_pred = self.lgb.predict_proba(X_selected)[:, 1]
        cb_pred = self.cb.predict_proba(X_selected)[:, 1]
        
        # Create meta-features
        meta_features = np.column_stack([xgb_pred, lgb_pred, cb_pred])
        
        # Get final prediction
        return self.meta.predict_proba(meta_features)[:, 1]
    
    def predict_binary(self, X, threshold=0.5):
        """Get binary predictions."""
        return (self.predict(X) >= threshold).astype(int)

# Instantiate predictor
predictor = SimpleEnsamblePredictor(
    {'xgb': xgb_model, 'lgb': lgb_model, 'cb': cb_model},
    meta_model,
    selected_indices
)

print("✓ Inference wrapper created")
print(f"  - Base models: 3 (XGBoost, LightGBM, CatBoost)")
print(f"  - Meta-learner: XGBoost")
print(f"  - Input features: {len(selected_features)}")

## Section 6: Inference and Predictions

In [ ]:
print("=== Single Sample Inference ===")

# Get a sample
sample_idx = 0
sample = X_test_selected[sample_idx:sample_idx+1]
sample_label = y_test[sample_idx]

# Make prediction
pred_proba = predictor.predict(sample_selected[np.newaxis, :])[0]
pred_binary = predictor.predict_binary(X_test_selected[sample_idx:sample_idx+1])[0]

print(f"\nSample #{sample_idx}:")
print(f"  - Prediction probability: {pred_proba:.4f}")
print(f"  - Binary prediction: {'FRAUD' if pred_binary == 1 else 'LEGITIMATE'}")
print(f"  - True label: {'FRAUD' if sample_label == 1 else 'LEGITIMATE'}")
print(f"  - Prediction correct: {'✓ YES' if pred_binary == sample_label else '✗ NO'}")

In [ ]:
# Batch inference
print("\n=== Batch Inference ===")

# Make predictions on test set
batch_pred_proba = predictor.predict(X_test_selected)
batch_pred_binary = predictor.predict_binary(X_test_selected)

# Statistics
print(f"\nBatch size: {len(batch_pred_proba):,}")
print(f"\nPrediction distribution:")
print(f"  - Predicted Legitimate (< 0.5): {(batch_pred_proba < 0.5).sum():,} ({(batch_pred_proba < 0.5).mean()*100:.2f}%)")
print(f"  - Predicted Fraud (≥ 0.5): {(batch_pred_proba >= 0.5).sum():,} ({(batch_pred_proba >= 0.5).mean()*100:.2f}%)")

print(f"\nPrediction confidence:")
print(f"  - Mean: {batch_pred_proba.mean():.4f}")
print(f"  - Std: {batch_pred_proba.std():.4f}")
print(f"  - Min: {batch_pred_proba.min():.4f}")
print(f"  - Max: {batch_pred_proba.max():.4f}")
print(f"  - Median: {np.median(batch_pred_proba):.4f}")

In [ ]:
# Prediction confidence visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Histogram
axes[0].hist(batch_pred_proba[y_test == 0], bins=50, alpha=0.6, label='Legitimate', color='green')
axes[0].hist(batch_pred_proba[y_test == 1], bins=50, alpha=0.6, label='Fraudulent', color='red')
axes[0].axvline(0.5, color='black', linestyle='--', label='Decision Threshold')
axes[0].set_xlabel('Fraud Probability')
axes[0].set_ylabel('Count')
axes[0].set_title('Prediction Confidence Distribution')
axes[0].legend()

# ROC Curve
from sklearn.metrics import roc_curve
fpr, tpr, thresholds = roc_curve(y_test, batch_pred_proba)
axes[1].plot(fpr, tpr, linewidth=2, label=f'AUC = {roc_auc_score(y_test, batch_pred_proba):.4f}')
axes[1].plot([0, 1], [0, 1], linestyle='--', color='gray', label='Random')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Prediction visualization complete")

## Section 7: Monitoring and Logging

In [ ]:
print("=== Performance Monitoring ===")

# Create monitoring report
monitoring_report = {
    'timestamp': pd.Timestamp.now().isoformat(),
    'inference_count': len(batch_pred_proba),
    'model_version': '1.0.0',
    'metrics': results['Ensemble'],
    'prediction_distribution': {
        'fraud_count': int((batch_pred_binary == 1).sum()),
        'fraud_rate': float((batch_pred_binary == 1).mean()),
        'legitimate_count': int((batch_pred_binary == 0).sum()),
        'legitimate_rate': float((batch_pred_binary == 0).mean())
    },
    'confidence_metrics': {
        'mean_probability': float(batch_pred_proba.mean()),
        'std_probability': float(batch_pred_proba.std()),
        'min_probability': float(batch_pred_proba.min()),
        'max_probability': float(batch_pred_proba.max())
    }
}

print("\nMonitoring Report:")
print(json.dumps(monitoring_report, indent=2))

# Save report
os.makedirs('results', exist_ok=True)
with open('results/monitoring_report.json', 'w') as f:
    json.dump(monitoring_report, f, indent=2)

print(f"\n✓ Monitoring report saved to results/monitoring_report.json")

In [ ]:
# Model drift detection
print("\n=== Data Drift Detection ===")

# Compare training vs test set statistics
print("\nFeature distribution comparison (first 5 features):")
for i in range(min(5, X_train_selected.shape[1])):
    train_mean = X_train_selected[:, i].mean()
    test_mean = X_test_selected[:, i].mean()
    train_std = X_train_selected[:, i].std()
    test_std = X_test_selected[:, i].std()
    
    pct_change = abs(test_mean - train_mean) / (train_mean + 1e-10) * 100
    
    print(f"\n  Feature {i}:")
    print(f"    Train: mean={train_mean:.4f}, std={train_std:.4f}")
    print(f"    Test:  mean={test_mean:.4f}, std={test_std:.4f}")
    print(f"    Change: {pct_change:.2f}%")

from scipy import stats

# Kolmogorov-Smirnov test for distribution shift
print("\n\nDistribution shift detection (K-S test, first 5 features):")
for i in range(min(5, X_train_selected.shape[1])):
    stat, pval = stats.ks_2samp(X_train_selected[:, i], X_test_selected[:, i])
    is_shifted = "✓ SHIFT" if pval < 0.05 else "✗ NO SHIFT"
    print(f"  Feature {i}: KS-stat={stat:.4f}, p-value={pval:.4f} {is_shifted}")

In [ ]:
print("\n=== Summary ===")
print(f"""
✓ WORKFLOW COMPLETE

Features Demonstrated:
  1. ✓ Data loading and exploration
  2. ✓ Data validation and quality checks
  3. ✓ Preprocessing and class balancing (SMOTE)
  4. ✓ Feature selection with SHAP ({len(selected_features)} features selected)
  5. ✓ Model training (XGBoost, LightGBM, CatBoost)
  6. ✓ Ensemble meta-learner
  7. ✓ Comprehensive evaluation
  8. ✓ Single and batch inference
  9. ✓ Model deployment configuration
  10. ✓ Performance monitoring
  11. ✓ Data drift detection

Best Model Performance:
  - Accuracy: {results['Ensemble']['accuracy']:.4f}
  - Precision: {results['Ensemble']['precision']:.4f}
  - Recall: {results['Ensemble']['recall']:.4f}
  - F1-Score: {results['Ensemble']['f1']:.4f}
  - AUC-ROC: {results['Ensemble']['auc']:.4f}

Next Steps:
  - Deploy API: make serve
  - Launch UI: streamlit run ui/fraud_app.py
  - Start MLflow: make mlflow-ui
  - Run Docker: make docker-full
  - See full workflow: DEMO_WORKFLOW.md
""")